# MeshAnything V1 & V2: 3D Mesh Generation and Analysis

## Project Overview
This notebook provides a complete pipeline to generate, visualize, and evaluate 3D meshes using **MeshAnything** (covering both V1 and V2 architectures). Furthermore, it implements a detailed quantitative analysis, comparing the generated meshes against Ground Truth 3D models (Rodin) using advanced spatial metrics such as Chamfer Distance (CD), Edge Chamfer Distance (ECD), and Normal Consistency (NC).

## Requirements to Run
To successfully execute this notebook, please ensure the following requirements are met:
1. **Google Drive:** The script requires Google Drive to be mounted. It uses your Drive to save generated 3D files (`.obj`) and 2D renderings (`.png`), as well as to load the Ground Truth reference models for the metrics comparison.
2. **GPU Acceleration:** For reasonable inference times during 3D generation, a GPU is strongly required. Please go to `Runtime` > `Change runtime type` > `Hardware accelerator` and select `T4 GPU` (or better).
3. **Network Access:** The script will automatically clone the required GitHub repositories and download the necessary PyTorch weights during execution.

## Execution Environment
This notebook was specifically designed, developed, and executed within the **Google Colab** environment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# 1. Clone repository if it doesn't exist yet
if not os.path.exists('/content/MeshAnythingV2'):
    !git clone https://github.com/buaacyw/MeshAnythingV2.git

# 2. Install Python 3.10 and create a virtual environment (venv) to replicate the original environment
!apt-get update -y
!apt-get install -y python3.10 python3.10-dev python3.10-venv
!python3.10 -m venv /content/env_mesh

# 3. Install PyTorch 2.1.1 with CUDA 11.8 inside the venv
!/content/env_mesh/bin/pip install pip --upgrade
!/content/env_mesh/bin/pip install torch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 --index-url https://download.pytorch.org/whl/cu118

# 4. Install MeshAnythingV2 and Gradio dependencies
!/content/env_mesh/bin/pip install -r /content/MeshAnythingV2/requirements.txt
!/content/env_mesh/bin/pip install gradio

# 5. Install flash-attn via pre-compiled wheel (no compilation, immediate installation!)
!/content/env_mesh/bin/pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.6/flash_attn-2.5.6+cu118torch2.1cxx11abiFALSE-cp310-cp310-linux_x86_64.whl

In [ ]:
# 6. Run the application (V2) from inside the virtual environment (with MPLBACKEND=Agg to prevent matplotlib errors)
%cd /content/MeshAnythingV2
!MPLBACKEND=Agg GRADIO_SHARE=True /content/env_mesh/bin/python app.py

### 7. Save Results to Google Drive (V2)
Execute this cell **after** you have used the Gradio interface and stopped the cell above. The code below will search for `.obj` and `.png` files in the output folder and copy them to the `anythingv2results` folder on your Google Drive for future comparison.

In [ ]:
import os
import shutil

# Define the destination directory in Google Drive for V2 results
dest_dir = '/content/drive/MyDrive/anythingv2results'
os.makedirs(dest_dir, exist_ok=True)
print(f"Destination folder ready: {dest_dir}")

work_dir = '/content/MeshAnythingV2'
arquivos_copiados = 0
print("Searching for generated .obj and .png files...")

# Traverse the entire MeshAnythingV2 directory
for root, dirs, files in os.walk(work_dir):
    # Ignore hidden or virtual environment folders to focus on results
    if '.git' in root or 'env_mesh' in root or '__pycache__' in root:
        continue

    for file in files:
        if file.endswith('.obj') or file.endswith('.png'):
            origem = os.path.join(root, file)
            destino = os.path.join(dest_dir, file)
            shutil.copy2(origem, destino)
            print(f"Copied: {file}")
            arquivos_copiados += 1

if arquivos_copiados > 0:
    print(f"\nSuccess! {arquivos_copiados} files were permanently saved in {dest_dir}")
else:
    print("\nNo .obj or .png files were found in the output folders.")

### 8. Visualize Renderings (Standard MeshAnything V2)
This cell displays the combined images generated by the repository itself, showing the model from various angles.

In [ ]:
import os
from IPython.display import Image, display

dest_dir = '/content/drive/MyDrive/anythingv2results'

# Filtrar as imagens geradas pela aplicação (as renderizações combinadas)
imagens = [f for f in os.listdir(dest_dir) if f.endswith('.png')]

if not imagens:
    print("Nenhuma imagem PNG encontrada na pasta de resultados.")
else:
    print(f"Encontradas {len(imagens)} imagens. Exibindo os resultados:\n")
    for img_name in imagens:
        print(f"Visualizando: {img_name}")
        caminho_img = os.path.join(dest_dir, img_name)
        display(Image(filename=caminho_img))
        print("-" * 50)

### 9. Visualize 3D Models (.obj)
This cell installs the necessary library to load 3D meshes and renders an interactive viewer for each generated model.

In [ ]:
!pip install -q open3d plotly

import os
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

dest_dir = '/content/drive/MyDrive/anythingv2results'

# Filtrar os arquivos .obj
obj_files = [f for f in os.listdir(dest_dir) if f.endswith('.obj')]

if not obj_files:
    print("Nenhum arquivo OBJ encontrado na pasta de resultados.")
else:
    print(f"Encontrados {len(obj_files)} arquivos OBJ. Processando com Open3D...\n")
    for obj_file in obj_files:
        print(f"Carregando modelo: {obj_file}")
        caminho_obj = os.path.join(dest_dir, obj_file)

        try:
            # 1. Carregar a malha usando o leitor do Open3D
            mesh = o3d.io.read_triangle_mesh(caminho_obj)

            # 2. Processamento do Open3D: Calcular normais de vértices para melhor iluminação
            mesh.compute_vertex_normals()

            # 3. Extrair os dados processados para NumPy array
            vertices = np.asarray(mesh.vertices)
            faces = np.asarray(mesh.triangles)

            # 4. Renderizar o objeto inline (já que janelas nativas do Open3D não abrem no Colab)
            fig = go.Figure(data=[
                go.Mesh3d(
                    x=vertices[:, 0],
                    y=vertices[:, 1],
                    z=vertices[:, 2],
                    i=faces[:, 0],
                    j=faces[:, 1],
                    k=faces[:, 2],
                    color='lightgreen', # Cor verde para diferenciar da célula anterior
                    opacity=0.9,
                    lighting=dict(ambient=0.4, diffuse=0.6, roughness=0.9, specular=0.5, fresnel=0.2)
                )
            ])

            fig.update_layout(
                title=f"Open3D Parse: {obj_file}",
                scene=dict(
                    xaxis=dict(visible=False),
                    yaxis=dict(visible=False),
                    zaxis=dict(visible=False),
                    aspectmode='data'
                ),
                margin=dict(l=0, r=0, b=0, t=40)
            )

            fig.show()

        except Exception as e:
            print(f"Erro ao processar {obj_file} com Open3D: {e}")
        print("-" * 50)


### 11. Analyze Topology (Wireframe)
This cell extracts the edges of the 3D mesh generating a **Wireframe** viewer, ideal for evaluating the quality and distribution of polygons generated by MeshAnything.

## Metrics Comparison: MeshAnything V1 vs V2 vs Ground Truth (Rodin)

This section implements the suggested metrics to compare the quality of the meshes generated by the V1 and V2 versions of MeshAnything against a "Ground Truth" mesh (Rodin).

**Calculated Metrics:**
*   **CD (Chamfer Distance):** Measures the average distance between points of two surfaces.
*   **NC (Normal Consistency):** Evaluates the similarity of point normal orientations between the two meshes.
*   **#V (Number of Vertices):** Total count of vertices in the mesh.
*   **#F (Number of Faces):** Total count of faces (triangles) in the mesh.
*   **V_Ratio (Vertices Ratio):** `#V` of the generated mesh / `#V` of the Ground Truth.
*   **F_Ratio (Faces Ratio):** `#F` of the generated mesh / `#F` of the Ground Truth.

We will sample 100,000 points from each mesh for the CD and NC calculations.

In [ ]:
import os
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

dest_dir = '/content/drive/MyDrive/anythingv2results'
obj_files = [f for f in os.listdir(dest_dir) if f.endswith('.obj')]

if not obj_files:
    print("Nenhum arquivo OBJ encontrado na pasta de resultados.")
else:
    print(f"Gerando visualização em Wireframe para {len(obj_files)} modelos...\n")
    for obj_file in obj_files:
        print(f"Analisando topologia: {obj_file}")
        caminho_obj = os.path.join(dest_dir, obj_file)

        try:
            # 1. Carregar a malha e processar vértices
            mesh = o3d.io.read_triangle_mesh(caminho_obj)
            mesh.compute_vertex_normals()
            vertices = np.asarray(mesh.vertices)
            faces = np.asarray(mesh.triangles)

            # 2. Extrair o Wireframe (Linhas) usando LineSet do Open3D
            lineset = o3d.geometry.LineSet.create_from_triangle_mesh(mesh)
            points = np.asarray(lineset.points)
            lines = np.asarray(lineset.lines)

            # 3. Preparar coordenadas estruturadas para desenhar linhas no Plotly (separadas por None)
            x_lines, y_lines, z_lines = [], [], []
            for line in lines:
                x_lines.extend([points[line[0], 0], points[line[1], 0], None])
                y_lines.extend([points[line[0], 1], points[line[1], 1], None])
                z_lines.extend([points[line[0], 2], points[line[1], 2], None])

            # 4. Criar a figura combinando o Sólido e o Wireframe
            fig = go.Figure()

            # Camada 1: Superfície semitransparente (Cor alaranjada)
            fig.add_trace(go.Mesh3d(
                x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
                i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                color='#e67e22', # Laranja/Clay
                opacity=0.4, # Mais transparente para destacar as linhas
                lighting=dict(ambient=0.5, diffuse=0.5, roughness=0.8, specular=0.2),
                name='Superfície'
            ))

            # Camada 2: Arestas / Topologia (Wireframe)
            fig.add_trace(go.Scatter3d(
                x=x_lines, y=y_lines, z=z_lines,
                mode='lines',
                line=dict(color='black', width=2),
                name='Topologia (Wireframe)',
                hoverinfo='none'
            ))

            fig.update_layout(
                title=f"Topologia: {obj_file}",
                scene=dict(
                    xaxis=dict(visible=False),
                    yaxis=dict(visible=False),
                    zaxis=dict(visible=False),
                    aspectmode='data'
                ),
                margin=dict(l=0, r=0, b=0, t=40),
                showlegend=True
            )

            fig.show()

        except Exception as e:
            print(f"Erro ao processar wireframe de {obj_file}: {e}")
        print("-" * 50)


---
# 🟢 PART 2: MeshAnything V1 (Baseline)
In this section, we will isolate a new environment to download, install, and run the **original MeshAnything (V1)**. The goal is to generate 3D meshes to serve as a *baseline* and compare them side-by-side with the improvements generated by V2 at the top of this notebook.

### 1. Prepare Environment (MeshAnything V1)
Let's clone the V1 repository and create a new specific virtual environment so it doesn't interfere with V2.

In [ ]:
import os

# 1. Clonar repositório da V1 caso ainda não exista
if not os.path.exists('/content/MeshAnythingV1'):
    !git clone https://github.com/buaacyw/MeshAnything.git /content/MeshAnythingV1

# 2. Instalar o pacote venv e criar um novo ambiente virtual isolado para a V1
!apt-get update -y > /dev/null
!apt-get install -y python3.10-venv > /dev/null
!python3.10 -m venv /content/env_mesh_v1
print("Repositório clonado e ambiente env_mesh_v1 criado!")

### 2. Install V1 Dependencies
Below, we install PyTorch, the libraries from the V1 `requirements.txt`, and Flash Attention (via wheel).

In [ ]:
# 3. Atualizar pip e instalar PyTorch compatível com CUDA 11.8 no novo venv
!/content/env_mesh_v1/bin/pip install pip --upgrade
!/content/env_mesh_v1/bin/pip install torch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 --index-url https://download.pytorch.org/whl/cu118

# 4. Instalar as dependências do repositório da V1
!/content/env_mesh_v1/bin/pip install -r /content/MeshAnythingV1/requirements.txt
!/content/env_mesh_v1/bin/pip install gradio

# 5. Instalar o flash-attn através do wheel pré-compilado para rapidez
!/content/env_mesh_v1/bin/pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.6/flash_attn-2.5.6+cu118torch2.1cxx11abiFALSE-cp310-cp310-linux_x86_64.whl

print("Instalação das dependências da V1 concluída!")

In [ ]:
# Correção: Fazer o downgrade do numpy para a versão 1.x para evitar o erro 'numpy.core.multiarray failed to import'
!/content/env_mesh_v1/bin/pip install "numpy<2"

### 3. Run Gradio (MeshAnything V1)
Run this cell to start the V1 interface. After generating your 3D model through the interface, **stop the execution of this cell**.

In [ ]:
# 6. Rodar a aplicação da V1 de dentro do seu respectivo ambiente virtual
%cd /content/MeshAnythingV1
!MPLBACKEND=Agg GRADIO_SHARE=True /content/env_mesh_v1/bin/python app.py

### 4. Save Results to Google Drive (V1 Baseline)
This cell searches for the generated files (`.obj` and `.png`) in the V1 folder and copies them to your Drive folder `anythingv1results`, ensuring the permanent storage of our baseline.

In [ ]:
import os
import shutil

# Definir o diretório de destino no Google Drive para os resultados da V1
dest_dir_v1 = '/content/drive/MyDrive/anythingv1results'
os.makedirs(dest_dir_v1, exist_ok=True)
print(f"Pasta de destino pronta: {dest_dir_v1}")

work_dir_v1 = '/content/MeshAnythingV1'
arquivos_copiados = 0
print("Procurando por arquivos .obj e .png gerados pela V1 (ignorando exemplos)...")

# Percorrer o diretório do MeshAnythingV1 inteiro
for root, dirs, files in os.walk(work_dir_v1):
    # Ignorar pastas ocultas, ambientes virtuais e a pasta de EXEMPLOS do repositório
    if '.git' in root or 'env_mesh_v1' in root or '__pycache__' in root or 'examples' in root:
        continue

    for file in files:
        if file.endswith('.obj') or file.endswith('.png'):
            origem = os.path.join(root, file)
            destino = os.path.join(dest_dir_v1, file)
            shutil.copy2(origem, destino)
            print(f"Copiado: {file}")
            arquivos_copiados += 1

if arquivos_copiados > 0:
    print(f"\nSucesso! {arquivos_copiados} arquivos foram salvos permanentemente em {dest_dir_v1}")
else:
    print("\nNenhum arquivo .obj ou .png foi encontrado nas pastas de saída da V1.")

### 5. Visualize 2D Renderings (V1)
Displays the combined images (Preview) generated by V1.

In [ ]:
import os
from IPython.display import Image, display

dest_dir_v1 = '/content/drive/MyDrive/anythingv1results'

# Filtrar as imagens geradas pela aplicação
imagens = [f for f in os.listdir(dest_dir_v1) if f.endswith('.png')]

if not imagens:
    print("Nenhuma imagem PNG encontrada na pasta de resultados da V1.")
else:
    print(f"Encontradas {len(imagens)} imagens. Exibindo os resultados:\n")
    for img_name in imagens:
        print(f"Visualizando (V1): {img_name}")
        caminho_img = os.path.join(dest_dir_v1, img_name)
        display(Image(filename=caminho_img))
        print("-" * 50)

### 6. Analyze 3D Topology (Wireframe V1)
We apply exactly the same Orange Wireframe style (#e67e22) superimposed on the V1 meshes to visually compare the topology efficiency (polygon count and distribution) compared to V2.

In [ ]:
import os
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

dest_dir_v1 = '/content/drive/MyDrive/anythingv1results'
obj_files = [f for f in os.listdir(dest_dir_v1) if f.endswith('.obj')]

if not obj_files:
    print("No OBJ files found in the V1 results folder.")
else:
    print(f"Generating Wireframe visualization for {len(obj_files)} V1 models...\n")
    for obj_file in obj_files:
        print(f"Analyzing topology (Baseline V1): {obj_file}")
        caminho_obj = os.path.join(dest_dir_v1, obj_file)

        try:
            # 1. Load the mesh and process vertices
            mesh = o3d.io.read_triangle_mesh(caminho_obj)
            mesh.compute_vertex_normals()
            vertices = np.asarray(mesh.vertices)
            faces = np.asarray(mesh.triangles)

            # 2. Extract the Wireframe (Lines) using Open3D's LineSet
            lineset = o3d.geometry.LineSet.create_from_triangle_mesh(mesh)
            points = np.asarray(lineset.points)
            lines = np.asarray(lineset.lines)

            # 3. Prepare structured coordinates to draw lines in Plotly (separated by None)
            x_lines, y_lines, z_lines = [], [], []
            for line in lines:
                x_lines.extend([points[line[0], 0], points[line[1], 0], None])
                y_lines.extend([points[line[0], 1], points[line[1], 1], None])
                z_lines.extend([points[line[0], 2], points[line[1], 2], None])

            # 4. Create the figure combining the Solid and the Wireframe
            fig = go.Figure()

            # Layer 1: Semi-transparent surface (Standard Orange Color)
            fig.add_trace(go.Mesh3d(
                x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
                i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                color='#e67e22', # Orange/Clay for consistency
                opacity=0.4,
                lighting=dict(ambient=0.5, diffuse=0.5, roughness=0.8, specular=0.2),
                name='Surface (V1)'
            ))

            # Layer 2: Edges / Topology (Black Wireframe)
            fig.add_trace(go.Scatter3d(
                x=x_lines, y=y_lines, z=z_lines,
                mode='lines',
                line=dict(color='black', width=2),
                name='Topology (Wireframe V1)',
                hoverinfo='none'
            ))

            fig.update_layout(
                title=f"V1 Topology: {obj_file}",
                scene=dict(
                    xaxis=dict(visible=False),
                    yaxis=dict(visible=False),
                    zaxis=dict(visible=False),
                    aspectmode='data'
                ),
                margin=dict(l=0, r=0, b=0, t=40),
                showlegend=True
            )

            fig.show()

        except Exception as e:
            print(f"Error processing wireframe for {obj_file}: {e}")
        print("-" * 50)

---
## 🟢 PART 3: Metrics Comparison (MeshAnything V1 vs V2 vs Ground Truth)

This section implements the suggested metrics to compare the quality of the meshes generated by the V1 and V2 versions of MeshAnything against a "Ground Truth" mesh (Rodin).

**Calculated Metrics:**
*   **CD (Chamfer Distance):** Measures the average distance between points of two surfaces.
*   **NC (Normal Consistency):** Evaluates the similarity of point normal orientations between the two meshes.
*   **#V (Number of Vertices):** Total count of vertices in the mesh.
*   **#F (Number of Faces):** Total count of faces (triangles) in the mesh.
*   **V_Ratio (Vertices Ratio):** `#V` of the generated mesh / `#V` of the Ground Truth.
*   **F_Ratio (Faces Ratio):** `#F` of the generated mesh / `#F` of the Ground Truth.

We will sample 100,000 points from each mesh for the CD and NC calculations.

In [ ]:
!pip install -q open3d pandas plotly trimesh


In [ ]:
import os
import open3d as o3d
import numpy as np
import pandas as pd
import trimesh

# --- Configurações ---

# Diretório onde os arquivos Ground Truth estão salvos
GT_DIR = '/content/drive/MyDrive/ground_truth'

# Número de pontos a serem amostrados para CD e NC
NUM_SAMPLE_POINTS = 100000
NUM_ECD_SAMPLE_POINTS = 10000 # Amostragem para arestas agudas

dest_dir_v1 = '/content/drive/MyDrive/anythingv1results'
dest_dir_v2 = '/content/drive/MyDrive/anythingv2results'

# Dicionário de mapeamento explícito para GT
GT_MAP = {
    'arthur': 'arthur_rodin.obj',
    'cannon': 'cannon_rodin.obj',
    'lion': 'lion_rodin.obj',
    'post': 'post_rodin.obj'
}

# --- Funções para cálculo das métricas ---

def load_and_preprocess_mesh(filepath):
    """Carrega uma malha OBJ e computa as normais."""
    if not os.path.exists(filepath):
        print(f"Erro: Arquivo não encontrado em {filepath}")
        return None
    try:
        mesh = o3d.io.read_triangle_mesh(filepath)
        if not mesh.has_vertices():
            print(f"Aviso: Malha vazia em {filepath}")
            return None
        mesh.compute_vertex_normals()
        return mesh
    except Exception as e:
        print(f"Erro ao carregar/pré-processar {filepath}: {e}")
        return None

def calculate_chamfer_distance(mesh_gt, mesh_gen, num_points):
    """Calcula a Chamfer Distance (Squared) entre duas malhas."""
    if mesh_gt is None or mesh_gen is None:
        return np.nan

    try:
        pcd_gt = mesh_gt.sample_points_uniformly(number_of_points=num_points)
        pcd_gen = mesh_gen.sample_points_uniformly(number_of_points=num_points)

        if len(pcd_gt.points) == 0 or len(pcd_gen.points) == 0:
            return np.nan

        # Distância (ao quadrado) de gen para gt
        dists_gen_to_gt = np.asarray(pcd_gen.compute_point_cloud_distance(pcd_gt))
        cd1 = np.mean(dists_gen_to_gt ** 2)

        # Distância (ao quadrado) de gt para gen
        dists_gt_to_gen = np.asarray(pcd_gt.compute_point_cloud_distance(pcd_gen))
        cd2 = np.mean(dists_gt_to_gen ** 2)

        return (cd1 + cd2) / 2
    except Exception as e:
        print(f"Erro no cálculo de Chamfer Distance: {e}")
        return np.nan

def calculate_normal_consistency(mesh_gt, mesh_gen, num_points):
    """Calcula a Normal Consistency bidirecional entre duas malhas."""
    if mesh_gt is None or mesh_gen is None:
        return np.nan

    try:
        pcd_gt = mesh_gt.sample_points_uniformly(number_of_points=num_points)
        pcd_gen = mesh_gen.sample_points_uniformly(number_of_points=num_points)

        if len(pcd_gt.points) == 0 or len(pcd_gen.points) == 0:
            return np.nan

        pcd_gt_tree = o3d.geometry.KDTreeFlann(pcd_gt)
        pcd_gen_tree = o3d.geometry.KDTreeFlann(pcd_gen)

        # gen -> gt
        dot_products_gen_to_gt = []
        for i in range(len(pcd_gen.points)):
            [k, idx, _] = pcd_gt_tree.search_knn_vector_3d(pcd_gen.points[i], 1)
            if k > 0:
                dot_products_gen_to_gt.append(np.abs(np.dot(pcd_gen.normals[i], pcd_gt.normals[idx[0]])))

        # gt -> gen
        dot_products_gt_to_gen = []
        for i in range(len(pcd_gt.points)):
            [k, idx, _] = pcd_gen_tree.search_knn_vector_3d(pcd_gt.points[i], 1)
            if k > 0:
                dot_products_gt_to_gen.append(np.abs(np.dot(pcd_gt.normals[i], pcd_gen.normals[idx[0]])))

        nc1 = np.mean(dot_products_gen_to_gt) if dot_products_gen_to_gt else np.nan
        nc2 = np.mean(dot_products_gt_to_gen) if dot_products_gt_to_gen else np.nan

        return (nc1 + nc2) / 2
    except Exception as e:
        print(f"Erro no cálculo de Normal Consistency: {e}")
        return np.nan

def get_sharp_edge_points(filepath, num_points, angle_thresh=np.pi/2):
    """Amostra pontos em arestas com ângulo diedro maior que o limite (padrão 90 graus)."""
    try:
        m = trimesh.load(filepath, force='mesh')
        if not hasattr(m, 'face_adjacency_angles'):
            return None

        sharp = m.face_adjacency_angles > angle_thresh
        sharp_edges = m.face_adjacency_edges[sharp]

        if len(sharp_edges) == 0:
            return np.zeros((0,3))

        v0 = m.vertices[sharp_edges[:, 0]]
        v1 = m.vertices[sharp_edges[:, 1]]

        lengths = np.linalg.norm(v1 - v0, axis=1)
        total_length = np.sum(lengths)
        if total_length == 0:
            return np.zeros((0,3))

        probs = lengths / total_length
        num_samples_per_edge = np.random.multinomial(num_points, probs)

        sampled_points = []
        for i, n in enumerate(num_samples_per_edge):
            if n > 0:
                t = np.random.rand(n, 1)
                pts = v0[i] + t * (v1[i] - v0[i])
                sampled_points.append(pts)

        if not sampled_points:
            return np.zeros((0,3))
        return np.vstack(sampled_points)
    except Exception as e:
        print(f"Erro ao amostrar arestas de {os.path.basename(filepath)}: {e}")
        return None

def calculate_ecd(gt_filepath, gen_filepath, num_points):
    """Calcula o Edge Chamfer Distance amostrando pontos nas arestas agudas."""
    pts_gt = get_sharp_edge_points(gt_filepath, num_points)
    pts_gen = get_sharp_edge_points(gen_filepath, num_points)

    if pts_gt is None or pts_gen is None or len(pts_gt)==0 or len(pts_gen)==0:
        return np.nan

    pcd_gt = o3d.geometry.PointCloud()
    pcd_gt.points = o3d.utility.Vector3dVector(pts_gt)
    pcd_gen = o3d.geometry.PointCloud()
    pcd_gen.points = o3d.utility.Vector3dVector(pts_gen)

    dists_gen_to_gt = np.asarray(pcd_gen.compute_point_cloud_distance(pcd_gt))
    dists_gt_to_gen = np.asarray(pcd_gt.compute_point_cloud_distance(pcd_gen))

    return (np.mean(dists_gen_to_gt**2) + np.mean(dists_gt_to_gen**2)) / 2

def get_mesh_stats(mesh):
    """Retorna o número de vértices e faces da malha."""
    if mesh is None:
        return np.nan, np.nan
    return len(mesh.vertices), len(mesh.triangles)

def get_gt_filepath(gen_filename):
    name_lower = gen_filename.lower()
    for key, gt_file in GT_MAP.items():
        if key in name_lower:
            return os.path.join(GT_DIR, gt_file)
    return None

# --- Processar e comparar malhas geradas ---

results = []
gt_cache = {}

def process_meshes_in_dir(directory, version_tag):
    global results
    if not os.path.exists(directory):
        print(f"Aviso: Pasta não encontrada {directory}")
        return

    obj_files = [f for f in os.listdir(directory) if f.endswith('.obj')]

    if not obj_files:
        print(f"Nenhum arquivo OBJ encontrado na pasta {directory} para {version_tag}.")
        return

    print(f"\nIniciando análise para {version_tag} ({len(obj_files)} arquivos)...")
    for obj_file in obj_files:
        filepath = os.path.join(directory, obj_file)
        gen_mesh = load_and_preprocess_mesh(filepath)

        if gen_mesh is None:
            continue

        gen_num_vertices, gen_num_faces = get_mesh_stats(gen_mesh)

        cd, ecd, nc = np.nan, np.nan, np.nan
        gt_num_vertices, gt_num_faces = np.nan, np.nan
        v_ratio, f_ratio = np.nan, np.nan

        gt_filepath = get_gt_filepath(obj_file)

        if gt_filepath and os.path.exists(gt_filepath):
            if gt_filepath not in gt_cache:
                print(f"  Carregando GT correspondente: {os.path.basename(gt_filepath)}")
                gt_cache[gt_filepath] = load_and_preprocess_mesh(gt_filepath)

            rodin_gt_mesh = gt_cache[gt_filepath]

            if rodin_gt_mesh is not None:
                gt_num_vertices, gt_num_faces = get_mesh_stats(rodin_gt_mesh)

                cd = calculate_chamfer_distance(rodin_gt_mesh, gen_mesh, NUM_SAMPLE_POINTS)
                nc = calculate_normal_consistency(rodin_gt_mesh, gen_mesh, NUM_SAMPLE_POINTS)
                ecd = calculate_ecd(gt_filepath, filepath, NUM_ECD_SAMPLE_POINTS)

                if not np.isnan(gt_num_vertices) and gt_num_vertices > 0:
                    v_ratio = gen_num_vertices / gt_num_vertices
                if not np.isnan(gt_num_faces) and gt_num_faces > 0:
                    f_ratio = gen_num_faces / gt_num_faces
        else:
            if gt_filepath is None:
                print(f"  Aviso: Nome de arquivo não reconhecido para mapeamento GT ({obj_file}).")
            else:
                print(f"  Aviso: Arquivo Ground Truth não encontrado em {gt_filepath}.")

        results.append({
            'Versão': version_tag,
            'Arquivo': obj_file,
            'CD': cd,
            'ECD': ecd,
            'NC': nc,
            '#V': gen_num_vertices,
            '#F': gen_num_faces,
            '#V_GT': gt_num_vertices,
            '#F_GT': gt_num_faces,
            'V_Ratio': v_ratio,
            'F_Ratio': f_ratio
        })
        print(f"  Processado {obj_file} ({version_tag})")

# Processar V1
process_meshes_in_dir(dest_dir_v1, 'MeshAnything V1')

# Processar V2
process_meshes_in_dir(dest_dir_v2, 'MeshAnything V2')

# Exibir resultados
if results:
    df_results = pd.DataFrame(results)
    df_results = df_results.round(6) # Aumentado para 6 pois Squared CD é bem menor
    df_results = df_results.sort_values(by=['Arquivo', 'Versão']).reset_index(drop=True)
    print("\n--- Resultados da Análise de Métricas ---")
    display(df_results)
else:
    print("Nenhum resultado gerado para comparação.")


In [ ]:
!pip install open3d trimesh pandas numpy scipy -q

In [ ]:
import os
import numpy as np
import pandas as pd
import open3d as o3d
import trimesh

from scipy.spatial import cKDTree

In [ ]:
# =========================
# CONFIGURAÇÕES
# =========================

GT_DIR = '/content/drive/MyDrive/ground_truth'

DEST_DIR_V1 = '/content/drive/MyDrive/anythingv1results'
DEST_DIR_V2 = '/content/drive/MyDrive/anythingv2results'

# Número de pontos para CD e NC
NUM_SAMPLE_POINTS = 100000

# Número de pontos para ECD
NUM_ECD_SAMPLE_POINTS = 10000

# Threshold para considerar uma aresta "aguda"
# Teste 45, 60 e 90 graus se quiser comparar sensibilidade.
SHARP_EDGE_ANGLE_DEG = 90

# Mapeamento entre nome do output e o respectivo GT do Rodin
GT_MAP = {
    'arthur': 'arthur_rodin.obj',
    'cannon': 'cannon_rodin.obj',
    'lion': 'lion_rodin.obj',
    'post': 'post_rodin.obj'
}

In [ ]:
def clean_open3d_mesh(mesh):
    """
    Limpa malha no Open3D:
    - remove vértices duplicados
    - remove triângulos duplicados
    - remove triângulos degenerados
    - remove vértices não referenciados
    - recalcula normais
    """
    if mesh is None:
        return None

    mesh.remove_duplicated_vertices()
    mesh.remove_duplicated_triangles()
    mesh.remove_degenerate_triangles()
    mesh.remove_unreferenced_vertices()
    mesh.compute_vertex_normals()

    return mesh


def normalize_open3d_mesh_to_unit_bbox(mesh):
    """
    Normaliza a malha para uma bounding box unitária.
    A malha é centralizada na origem e escalada pelo maior eixo da bbox.
    """
    if mesh is None:
        return None

    mesh_norm = o3d.geometry.TriangleMesh(mesh)

    bbox = mesh_norm.get_axis_aligned_bounding_box()
    center = bbox.get_center()
    extent = np.max(bbox.get_extent())

    mesh_norm.translate(-center)

    if extent > 0:
        mesh_norm.scale(1.0 / extent, center=(0, 0, 0))

    mesh_norm.compute_vertex_normals()
    return mesh_norm


def load_open3d_mesh(filepath, normalize=True, clean=True):
    """
    Carrega uma malha OBJ usando Open3D, limpa e normaliza.
    """
    if not os.path.exists(filepath):
        print(f"Erro: arquivo não encontrado: {filepath}")
        return None

    try:
        mesh = o3d.io.read_triangle_mesh(filepath)

        if not mesh.has_vertices():
            print(f"Aviso: malha vazia: {filepath}")
            return None

        mesh.compute_vertex_normals()

        if clean:
            mesh = clean_open3d_mesh(mesh)

        if normalize:
            mesh = normalize_open3d_mesh_to_unit_bbox(mesh)

        return mesh

    except Exception as e:
        print(f"Erro ao carregar {filepath}: {e}")
        return None


def get_mesh_stats_open3d(mesh):
    """
    Retorna número de vértices e faces da malha Open3D.
    """
    if mesh is None:
        return np.nan, np.nan

    return len(mesh.vertices), len(mesh.triangles)

In [ ]:
def load_trimesh_mesh(filepath, normalize=True, clean=True):
    """
    Carrega uma malha com trimesh, opcionalmente limpa e normaliza.
    """
    if not os.path.exists(filepath):
        print(f"Erro: arquivo não encontrado: {filepath}")
        return None

    try:
        mesh = trimesh.load(filepath, force='mesh', process=False)

        if mesh is None or mesh.vertices is None or len(mesh.vertices) == 0:
            print(f"Aviso: malha vazia em {filepath}")
            return None

        if clean:
            # Remove faces degeneradas
            try:
                mesh.update_faces(mesh.nondegenerate_faces())
            except Exception:
                pass

            # Remove vértices não usados
            try:
                mesh.remove_unreferenced_vertices()
            except Exception:
                pass

            # Remove faces duplicadas, se disponível
            try:
                unique_faces = np.unique(np.sort(mesh.faces, axis=1), axis=0)
                # reconstruir diretamente pode alterar ordem; em geral basta usar process=True,
                # mas mantemos simples para evitar quebrar o arquivo.
            except Exception:
                pass

        if normalize:
            vertices = mesh.vertices.copy()
            min_bound = vertices.min(axis=0)
            max_bound = vertices.max(axis=0)
            center = (min_bound + max_bound) / 2.0
            extent = np.max(max_bound - min_bound)

            vertices = vertices - center
            if extent > 0:
                vertices = vertices / extent

            mesh = trimesh.Trimesh(
                vertices=vertices,
                faces=mesh.faces.copy(),
                process=False
            )

        return mesh

    except Exception as e:
        print(f"Erro ao carregar com trimesh {filepath}: {e}")
        return None


def trimesh_diagnostics(filepath):
    """
    Retorna informações úteis sobre o OBJ com trimesh.
    """
    try:
        m = trimesh.load(filepath, force='mesh', process=False)

        if m is None or len(m.vertices) == 0:
            return {
                'watertight': np.nan,
                'components': np.nan,
                'degenerate_faces': np.nan,
                'euler_number': np.nan
            }

        face_areas = m.area_faces if hasattr(m, 'area_faces') else np.array([])
        degenerate_faces = int(np.sum(face_areas == 0)) if len(face_areas) > 0 else np.nan

        try:
            components = len(m.split(only_watertight=False))
        except Exception:
            components = np.nan

        return {
            'watertight': bool(m.is_watertight),
            'components': components,
            'degenerate_faces': degenerate_faces,
            'euler_number': m.euler_number if hasattr(m, 'euler_number') else np.nan
        }

    except Exception as e:
        print(f"Erro no diagnóstico de {filepath}: {e}")
        return {
            'watertight': np.nan,
            'components': np.nan,
            'degenerate_faces': np.nan,
            'euler_number': np.nan
        }

In [ ]:
def sample_points_from_mesh(mesh, num_points, use_triangle_normal=True):
    """
    Amostra pontos de uma malha Open3D.

    Usa use_triangle_normal=True quando disponível para obter normais mais estáveis.
    """
    try:
        pcd = mesh.sample_points_uniformly(
            number_of_points=num_points,
            use_triangle_normal=use_triangle_normal
        )
    except TypeError:
        # Compatibilidade com versões antigas do Open3D
        pcd = mesh.sample_points_uniformly(number_of_points=num_points)

    return pcd


def calculate_chamfer_distance_squared(mesh_gt, mesh_gen, num_points):
    """
    Calcula Chamfer Distance quadrática média bidirecional.

    CD = 0.5 * (mean(d(gen -> gt)^2) + mean(d(gt -> gen)^2))
    """
    if mesh_gt is None or mesh_gen is None:
        return np.nan

    try:
        pcd_gt = sample_points_from_mesh(mesh_gt, num_points)
        pcd_gen = sample_points_from_mesh(mesh_gen, num_points)

        pts_gt = np.asarray(pcd_gt.points)
        pts_gen = np.asarray(pcd_gen.points)

        if len(pts_gt) == 0 or len(pts_gen) == 0:
            return np.nan

        tree_gt = cKDTree(pts_gt)
        tree_gen = cKDTree(pts_gen)

        d_gen_to_gt, _ = tree_gt.query(pts_gen, k=1)
        d_gt_to_gen, _ = tree_gen.query(pts_gt, k=1)

        cd = 0.5 * (np.mean(d_gen_to_gt ** 2) + np.mean(d_gt_to_gen ** 2))

        return cd

    except Exception as e:
        print(f"Erro no cálculo de CD: {e}")
        return np.nan

In [ ]:
def ensure_point_cloud_normals(pcd):
    """
    Garante que o point cloud tenha normais.
    Se não tiver, estima normais.
    """
    if not pcd.has_normals():
        pcd.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamKNN(knn=30)
        )
        pcd.normalize_normals()

    return pcd


def calculate_normal_consistency(mesh_gt, mesh_gen, num_points):
    """
    Calcula Normal Consistency bidirecional.

    Para cada ponto de uma amostra, encontra o vizinho mais próximo na outra
    amostra e calcula |dot(n1, n2)|. O valor final é a média bidirecional.
    """
    if mesh_gt is None or mesh_gen is None:
        return np.nan

    try:
        pcd_gt = sample_points_from_mesh(mesh_gt, num_points, use_triangle_normal=True)
        pcd_gen = sample_points_from_mesh(mesh_gen, num_points, use_triangle_normal=True)

        pcd_gt = ensure_point_cloud_normals(pcd_gt)
        pcd_gen = ensure_point_cloud_normals(pcd_gen)

        pts_gt = np.asarray(pcd_gt.points)
        pts_gen = np.asarray(pcd_gen.points)

        normals_gt = np.asarray(pcd_gt.normals)
        normals_gen = np.asarray(pcd_gen.normals)

        if len(pts_gt) == 0 or len(pts_gen) == 0:
            return np.nan

        if len(normals_gt) != len(pts_gt) or len(normals_gen) != len(pts_gen):
            return np.nan

        tree_gt = cKDTree(pts_gt)
        tree_gen = cKDTree(pts_gen)

        _, idx_gen_to_gt = tree_gt.query(pts_gen, k=1)
        _, idx_gt_to_gen = tree_gen.query(pts_gt, k=1)

        dots_gen_to_gt = np.abs(
            np.sum(normals_gen * normals_gt[idx_gen_to_gt], axis=1)
        )

        dots_gt_to_gen = np.abs(
            np.sum(normals_gt * normals_gen[idx_gt_to_gen], axis=1)
        )

        nc = 0.5 * (np.mean(dots_gen_to_gt) + np.mean(dots_gt_to_gen))

        return nc

    except Exception as e:
        print(f"Erro no cálculo de Normal Consistency: {e}")
        return np.nan

In [ ]:
def get_sharp_edge_points_from_trimesh(mesh, num_points, angle_thresh_rad=np.pi/2):
    """
    Amostra pontos em arestas agudas de uma malha trimesh.

    Arestas agudas são definidas como arestas com ângulo diedro maior
    que angle_thresh_rad.
    """
    if mesh is None:
        return None

    try:
        if not hasattr(mesh, 'face_adjacency_angles'):
            return None

        if len(mesh.face_adjacency_angles) == 0:
            return np.zeros((0, 3))

        sharp_mask = mesh.face_adjacency_angles > angle_thresh_rad
        sharp_edges = mesh.face_adjacency_edges[sharp_mask]

        if len(sharp_edges) == 0:
            return np.zeros((0, 3))

        v0 = mesh.vertices[sharp_edges[:, 0]]
        v1 = mesh.vertices[sharp_edges[:, 1]]

        lengths = np.linalg.norm(v1 - v0, axis=1)
        total_length = np.sum(lengths)

        if total_length <= 0:
            return np.zeros((0, 3))

        probs = lengths / total_length
        num_samples_per_edge = np.random.multinomial(num_points, probs)

        sampled_points = []

        for i, n in enumerate(num_samples_per_edge):
            if n > 0:
                t = np.random.rand(n, 1)
                pts = v0[i] + t * (v1[i] - v0[i])
                sampled_points.append(pts)

        if len(sampled_points) == 0:
            return np.zeros((0, 3))

        return np.vstack(sampled_points)

    except Exception as e:
        print(f"Erro ao amostrar arestas agudas: {e}")
        return None


def calculate_edge_chamfer_distance_squared(
    gt_filepath,
    gen_filepath,
    num_points,
    angle_thresh_deg=90
):
    """
    Calcula ECD aproximado:
    - carrega GT e saída com trimesh
    - normaliza ambos para bounding box unitária
    - detecta arestas agudas por ângulo diedro
    - amostra pontos nessas arestas
    - calcula Chamfer Distance quadrática entre esses pontos
    """
    angle_thresh_rad = np.deg2rad(angle_thresh_deg)

    try:
        mesh_gt = load_trimesh_mesh(gt_filepath, normalize=True, clean=True)
        mesh_gen = load_trimesh_mesh(gen_filepath, normalize=True, clean=True)

        pts_gt = get_sharp_edge_points_from_trimesh(
            mesh_gt,
            num_points,
            angle_thresh_rad=angle_thresh_rad
        )

        pts_gen = get_sharp_edge_points_from_trimesh(
            mesh_gen,
            num_points,
            angle_thresh_rad=angle_thresh_rad
        )

        if pts_gt is None or pts_gen is None:
            return np.nan

        if len(pts_gt) == 0 or len(pts_gen) == 0:
            return np.nan

        tree_gt = cKDTree(pts_gt)
        tree_gen = cKDTree(pts_gen)

        d_gen_to_gt, _ = tree_gt.query(pts_gen, k=1)
        d_gt_to_gen, _ = tree_gen.query(pts_gt, k=1)

        ecd = 0.5 * (np.mean(d_gen_to_gt ** 2) + np.mean(d_gt_to_gen ** 2))

        return ecd

    except Exception as e:
        print(f"Erro no cálculo de ECD: {e}")
        return np.nan

In [ ]:
def get_gt_filepath(gen_filename):
    """
    Encontra o arquivo GT correspondente ao arquivo gerado.
    """
    name_lower = gen_filename.lower()

    for key, gt_file in GT_MAP.items():
        if key in name_lower:
            return os.path.join(GT_DIR, gt_file)

    return None


def list_obj_files(directory):
    """
    Lista arquivos .obj de um diretório.
    """
    if not os.path.exists(directory):
        print(f"Aviso: pasta não encontrada: {directory}")
        return []

    return sorted([
        f for f in os.listdir(directory)
        if f.lower().endswith('.obj')
    ])

In [ ]:
results = []
gt_cache = {}


def process_meshes_in_dir(directory, version_tag):
    """
    Processa todos os OBJ de um diretório e compara com seus GTs do Rodin.
    """
    obj_files = list_obj_files(directory)

    if len(obj_files) == 0:
        print(f"Nenhum arquivo OBJ encontrado em {directory} para {version_tag}.")
        return

    print(f"\nIniciando análise para {version_tag} ({len(obj_files)} arquivos)...")

    for obj_file in obj_files:
        gen_filepath = os.path.join(directory, obj_file)

        print(f"\nProcessando: {obj_file} ({version_tag})")

        # Carrega versão gerada sem normalizar, apenas para estatísticas originais
        gen_mesh_raw = load_open3d_mesh(
            gen_filepath,
            normalize=False,
            clean=False
        )

        gen_v_raw, gen_f_raw = get_mesh_stats_open3d(gen_mesh_raw)

        # Carrega versão gerada limpa e normalizada para métricas
        gen_mesh = load_open3d_mesh(
            gen_filepath,
            normalize=True,
            clean=True
        )

        gen_v_clean, gen_f_clean = get_mesh_stats_open3d(gen_mesh)

        diag_gen = trimesh_diagnostics(gen_filepath)

        cd = np.nan
        ecd = np.nan
        nc = np.nan

        gt_v_raw = np.nan
        gt_f_raw = np.nan
        gt_v_clean = np.nan
        gt_f_clean = np.nan

        v_retention = np.nan
        f_retention = np.nan

        v_reduction = np.nan
        f_reduction = np.nan

        gt_filepath = get_gt_filepath(obj_file)

        if gt_filepath is None:
            print(f"  Aviso: nome não reconhecido para mapear GT: {obj_file}")

        elif not os.path.exists(gt_filepath):
            print(f"  Aviso: GT não encontrado: {gt_filepath}")

        else:
            # GT raw para estatística original
            gt_mesh_raw = load_open3d_mesh(
                gt_filepath,
                normalize=False,
                clean=False
            )

            gt_v_raw, gt_f_raw = get_mesh_stats_open3d(gt_mesh_raw)

            # GT limpo e normalizado para métricas
            if gt_filepath not in gt_cache:
                print(f"  Carregando GT correspondente: {os.path.basename(gt_filepath)}")
                gt_cache[gt_filepath] = load_open3d_mesh(
                    gt_filepath,
                    normalize=True,
                    clean=True
                )

            gt_mesh = gt_cache[gt_filepath]
            gt_v_clean, gt_f_clean = get_mesh_stats_open3d(gt_mesh)

            if gt_mesh is not None and gen_mesh is not None:
                cd = calculate_chamfer_distance_squared(
                    gt_mesh,
                    gen_mesh,
                    NUM_SAMPLE_POINTS
                )

                nc = calculate_normal_consistency(
                    gt_mesh,
                    gen_mesh,
                    NUM_SAMPLE_POINTS
                )

                ecd = calculate_edge_chamfer_distance_squared(
                    gt_filepath,
                    gen_filepath,
                    NUM_ECD_SAMPLE_POINTS,
                    angle_thresh_deg=SHARP_EDGE_ANGLE_DEG
                )

                if gt_v_raw and gt_v_raw > 0:
                    v_retention = gen_v_clean / gt_v_raw
                    v_reduction = 1.0 - v_retention

                if gt_f_raw and gt_f_raw > 0:
                    f_retention = gen_f_clean / gt_f_raw
                    f_reduction = 1.0 - f_retention

        results.append({
            'Versão': version_tag,
            'Arquivo': obj_file,

            'CD_squared': cd,
            'ECD_squared': ecd,
            'NC': nc,

            '#V_raw': gen_v_raw,
            '#F_raw': gen_f_raw,
            '#V_clean': gen_v_clean,
            '#F_clean': gen_f_clean,

            '#V_GT_raw': gt_v_raw,
            '#F_GT_raw': gt_f_raw,
            '#V_GT_clean': gt_v_clean,
            '#F_GT_clean': gt_f_clean,

            'V_Retencao_Rodin': v_retention,
            'F_Retencao_Rodin': f_retention,
            'V_Reducao_%': v_reduction * 100 if not np.isnan(v_reduction) else np.nan,
            'F_Reducao_%': f_reduction * 100 if not np.isnan(f_reduction) else np.nan,

            'Watertight': diag_gen['watertight'],
            'Components': diag_gen['components'],
            'Degenerate_Faces': diag_gen['degenerate_faces'],
            'Euler_Number': diag_gen['euler_number']
        })

        print(f"  Finalizado: {obj_file}")

In [ ]:
results = []
gt_cache = {}

process_meshes_in_dir(DEST_DIR_V1, 'MeshAnything V1')
process_meshes_in_dir(DEST_DIR_V2, 'MeshAnything V2')

df_results = pd.DataFrame(results)

if len(df_results) > 0:
    df_results = df_results.sort_values(
        by=['Arquivo', 'Versão']
    ).reset_index(drop=True)

    display(df_results.round(6))
else:
    print("Nenhum resultado gerado.")

In [ ]:
cols_article = [
    'Arquivo',
    'Versão',
    'CD_squared',
    'ECD_squared',
    'NC',
    '#V_clean',
    '#F_clean',
    '#V_GT_raw',
    '#F_GT_raw',
    'V_Retencao_Rodin',
    'F_Retencao_Rodin',
    'V_Reducao_%',
    'F_Reducao_%'
]

df_article = df_results[cols_article].copy()

df_article = df_article.rename(columns={
    'CD_squared': 'CD',
    'ECD_squared': 'ECD',
    '#V_clean': '#V',
    '#F_clean': '#F',
    '#V_GT_raw': '#V_Rodin',
    '#F_GT_raw': '#F_Rodin',
    'V_Retencao_Rodin': 'V_Retencao',
    'F_Retencao_Rodin': 'F_Retencao'
})

display(df_article.round(6))

In [ ]:
def compare_v1_v2(df):
    """
    Gera uma tabela indicando qual versão venceu por métrica em cada objeto.
    Menor CD é melhor.
    Menor ECD é melhor.
    Maior NC é melhor.
    """
    rows = []

    for arquivo_base in sorted(set(df['Arquivo'].str.lower().str.replace('output_', '', regex=False))):
        pass

    # Melhor agrupar pelo nome do objeto usando GT_MAP
    for obj_key in GT_MAP.keys():
        df_obj = df[df['Arquivo'].str.lower().str.contains(obj_key)]

        if len(df_obj) < 2:
            continue

        v1 = df_obj[df_obj['Versão'] == 'MeshAnything V1']
        v2 = df_obj[df_obj['Versão'] == 'MeshAnything V2']

        if len(v1) == 0 or len(v2) == 0:
            continue

        v1 = v1.iloc[0]
        v2 = v2.iloc[0]

        rows.append({
            'Objeto': obj_key,

            'CD_V1': v1['CD_squared'],
            'CD_V2': v2['CD_squared'],
            'Melhor_CD': 'V1' if v1['CD_squared'] < v2['CD_squared'] else 'V2',

            'ECD_V1': v1['ECD_squared'],
            'ECD_V2': v2['ECD_squared'],
            'Melhor_ECD': 'V1' if v1['ECD_squared'] < v2['ECD_squared'] else 'V2',

            'NC_V1': v1['NC'],
            'NC_V2': v2['NC'],
            'Melhor_NC': 'V1' if v1['NC'] > v2['NC'] else 'V2',

            'Faces_V1': v1['#F_clean'],
            'Faces_V2': v2['#F_clean'],
            'Mais_faces': 'V1' if v1['#F_clean'] > v2['#F_clean'] else 'V2'
        })

    return pd.DataFrame(rows)


df_compare = compare_v1_v2(df_results)
display(df_compare.round(6))

In [ ]:
df_high_faces = df_results[df_results['#F_clean'] > 1600].copy()

if len(df_high_faces) == 0:
    print("Nenhum output com mais de 1600 faces após limpeza.")
else:
    print("Outputs com mais de 1600 faces após limpeza:")
    display(df_high_faces[['Versão', 'Arquivo', '#V_clean', '#F_clean', 'Components', 'Degenerate_Faces', 'Watertight']])

In [ ]:
def run_ecd_sensitivity(df_results, thresholds_deg=[45, 60, 90]):
    rows = []

    for _, row in df_results.iterrows():
        obj_file = row['Arquivo']

        if row['Versão'] == 'MeshAnything V1':
            gen_filepath = os.path.join(DEST_DIR_V1, obj_file)
        else:
            gen_filepath = os.path.join(DEST_DIR_V2, obj_file)

        gt_filepath = get_gt_filepath(obj_file)

        if gt_filepath is None or not os.path.exists(gt_filepath):
            continue

        result_row = {
            'Arquivo': obj_file,
            'Versão': row['Versão']
        }

        for th in thresholds_deg:
            ecd = calculate_edge_chamfer_distance_squared(
                gt_filepath,
                gen_filepath,
                NUM_ECD_SAMPLE_POINTS,
                angle_thresh_deg=th
            )
            result_row[f'ECD_{th}deg'] = ecd

        rows.append(result_row)

    return pd.DataFrame(rows)


df_ecd_sensitivity = run_ecd_sensitivity(df_results, thresholds_deg=[45, 60, 90])
display(df_ecd_sensitivity.round(6))

In [ ]:
df_article_clean = df_article[
    df_article["Arquivo"].str.startswith("output_")
].copy()

df_article_clean = df_article_clean.sort_values(
    by=["Arquivo", "Versão"]
).reset_index(drop=True)

display(df_article_clean.round(6))

In [ ]:
df_article_clean.to_csv(
    "/content/drive/MyDrive/meshanything_analysis_outputs/metricas_artigo_outputs_finais.csv",
    index=False
)